In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")


In [ ]:
import subprocess, sys, json, time, shutil
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ham10000-augmentation')
else:
    ROOT = Path.cwd()

DATA_DIR  = ROOT / 'data'
SPLIT_DIR = DATA_DIR / 'processed' / 'splits'
IMG_DIR   = DATA_DIR / 'HAM10000_images'
LORA_DIR  = ROOT / 'models' / 'lora_mel'
SYNTH_DIR = ROOT / 'synthetic' / 'lora'
LORA_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

# ── Hiperparámetros de entrenamiento ──────────────────────────────────────────
MODEL_ID          = 'runwayml/stable-diffusion-v1-5'
INSTANCE_PROMPT   = 'a dermoscopy image of sks melanoma skin lesion'
CLASS_PROMPT      = 'a dermoscopy image of a skin lesion'
LORA_RANK         = 32
MAX_TRAIN_STEPS   = 6000
LEARNING_RATE     = 1e-4
TEXT_ENCODER_LR   = 5e-5
PRIOR_LOSS_WEIGHT = 1.0
NUM_CLASS_IMAGES  = 800
CHECKPOINT_STEPS  = 500
SNR_GAMMA         = 5.0
SEED              = 42

# ── Generación ────────────────────────────────────────────────────────────────
GENERATE_N      = 4500
GENERATE_STEPS  = 30
GENERATE_BATCH  = 4
GENERATE_CFG    = 7.5
GENERATE_PROMPT = INSTANCE_PROMPT + ', high quality, sharp dermoscopic detail'
NEGATIVE_PROMPT = ('low quality, blurry, cartoon, illustration, text, '
                   'watermark, white background, jpeg artifact, oversaturated')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'ROOT   : {ROOT}')


In [ ]:
# ── Estado (sin cargar modelos — corre esto al reconectar) ───────────────────
lora_done = LORA_DIR / 'lora_done.txt'
gen_done  = SYNTH_DIR / 'generation_done.txt'
ckpts     = sorted(LORA_DIR.glob('checkpoint-*'))
last_ckpt = ckpts[-1].name if ckpts else 'ninguno'
n_synth   = len(list(SYNTH_DIR.glob('lora_*.jpg')))

print('=' * 58)
print(f'  Bloque 1 — Datos  : {"✅" if Path("/content/lora_data_ready.txt").exists() else "⬜ pendiente"}')
print(f'  Bloque 2 — LoRA   : {"✅ completo" if lora_done.exists() else f"⬜ pendiente  (último ckpt: {last_ckpt})"}')
print(f'  Bloque 3 — Genera : {"✅ completo" if gen_done.exists() else f"⬜ {n_synth}/{GENERATE_N} imágenes"}')
print('=' * 58)
if lora_done.exists():
    info = json.loads(lora_done.read_text())
    print(f'  LoRA: rank={info["rank"]}, steps={info["steps"]}, {info["elapsed_h"]}h')
if gen_done.exists():
    info = json.loads(gen_done.read_text())
    print(f'  Gen : {info["n"]} imgs, {info["num_inference_steps"]} steps, cfg={info["guidance_scale"]}')


In [ ]:
# ── Bloque 0 — Instalar dependencias ─────────────────────────────────────────
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

try:
    import diffusers
    ver = tuple(int(x) for x in diffusers.__version__.split('.')[:2])
    assert ver >= (0, 29), f'versión {diffusers.__version__} muy antigua'
    import peft, accelerate, xformers
    print(f'diffusers {diffusers.__version__} — listo')
except (ImportError, AssertionError) as e:
    print(f'Instalando paquetes ({e})...')
    pip('diffusers==0.29.2', 'peft==0.11.1', 'accelerate==0.30.1',
        'transformers==4.40.2', 'xformers', 'triton', 'datasets')
    print('✅ Instalado — reinicia el runtime si es la primera vez')


In [ ]:
# ── Bloque 1 — Preparar datos de entrenamiento ───────────────────────────────
import pandas as pd

INST_DIR    = Path('/content/lora_instance')   # melanoma  → concepto
CLASS_DIR   = Path('/content/lora_class')       # nv        → prior preservation
data_marker = Path('/content/lora_data_ready.txt')

if data_marker.exists():
    n_i = len(list(INST_DIR.glob('*.jpg')))
    n_c = len(list(CLASS_DIR.glob('*.jpg')))
    print(f'Datos listos: {n_i} instancia, {n_c} clase — saltando')
else:
    # Extraer zip
    zip_path  = ROOT / 'classification_data.zip'
    extracted = Path('/content/classification_data')
    if not extracted.exists():
        print('Extrayendo classification_data.zip...')
        subprocess.check_call(['unzip', '-q', str(zip_path), '-d', str(extracted)])

    # Localizar train.csv dinámicamente
    hits = list(extracted.rglob('train.csv'))
    if not hits:
        raise FileNotFoundError(
            f'No se encontró train.csv.\n'
            f'Contenido raíz: {[p.name for p in extracted.iterdir()]}'
        )
    split_base = hits[0].parent
    print(f'train.csv en: {split_base}')

    # Localizar carpeta de imágenes (el zip usa images/, no HAM10000_images/)
    img_candidates = [
        extracted / 'images',
        split_base.parent / 'images',
        split_base.parent / 'HAM10000_images',
    ]
    img_base = next((p for p in img_candidates if p.exists()), None)
    if img_base is None:
        raise FileNotFoundError(
            f'No se encontró carpeta de imágenes.\n'
            f'Carpetas disponibles: {[p for p in extracted.rglob("*") if p.is_dir()]}'
        )
    print(f'Imágenes en: {img_base}')

    train_df = pd.read_csv(split_base / 'train.csv')
    print(f'Columnas: {list(train_df.columns)}')

    # Detectar columna de imagen
    name_col = 'image_id' if 'image_id' in train_df.columns else 'image_path'

    # label puede ser string ('mel'/'nv') o numérico (1/0)
    mel_val = 1   if train_df['label'].dtype != object else 'mel'
    nv_val  = 0   if train_df['label'].dtype != object else 'nv'

    mel_rows = train_df[train_df['label'] == mel_val]
    nv_rows  = train_df[train_df['label'] == nv_val].sample(n=NUM_CLASS_IMAGES, random_state=SEED)
    print(f'mel: {len(mel_rows)}  |  nv (muestra): {len(nv_rows)}')

    def copy_images(rows, dst):
        dst.mkdir(parents=True, exist_ok=True)
        ok = 0
        for _, row in rows.iterrows():
            fname = Path(str(row[name_col])).name
            if not fname.endswith('.jpg'):
                fname += '.jpg'
            src = img_base / fname
            if src.exists():
                shutil.copy2(src, dst / fname)
                ok += 1
        return ok

    n_i = copy_images(mel_rows, INST_DIR)
    n_c = copy_images(nv_rows,  CLASS_DIR)
    data_marker.write_text('ok')
    print(f'✅ {n_i} melanoma → instancia   |   {n_c} nv → clase (prior preservation)')


In [ ]:
# ── Bloque 2 — Entrenar LoRA ──────────────────────────────────────────────────
# Resume automático: --resume_from_checkpoint=latest retoma desde el último
# checkpoint guardado en LORA_DIR (cada 500 steps). Si la sesión se cierra,
# vuelve a correr esta celda y continúa desde donde quedó.

if lora_done.exists():
    info = json.loads(lora_done.read_text())
    print(f'LoRA ya completo — {info.get("elapsed_h", "?")}h total   (saltando)')
else:
    # Descargar script de entrenamiento (DreamBooth-LoRA oficial de diffusers)
    script = Path('/content/train_dreambooth_lora.py')
    if not script.exists():
        import urllib.request
        url = ('https://raw.githubusercontent.com/huggingface/'
               'diffusers/v0.29.2/examples/dreambooth/train_dreambooth_lora.py')
        print('Descargando script...')
        urllib.request.urlretrieve(url, script)
        print('Script listo')

    # Config de accelerate sin prompt interactivo
    accel_cfg = Path.home() / '.cache' / 'huggingface' / 'accelerate' / 'default_config.yaml'
    accel_cfg.parent.mkdir(parents=True, exist_ok=True)
    accel_cfg.write_text(
        'compute_environment: LOCAL_MACHINE\n'
        'distributed_type: NO\n'
        'mixed_precision: fp16\n'
        'use_cpu: false\n'
        'num_processes: 1\n'
    )

    cmd = [
        'accelerate', 'launch', str(script),
        '--pretrained_model_name_or_path', MODEL_ID,
        '--instance_data_dir',            str(INST_DIR),
        '--class_data_dir',               str(CLASS_DIR),
        '--output_dir',                   str(LORA_DIR),
        '--instance_prompt',              INSTANCE_PROMPT,
        '--class_prompt',                 CLASS_PROMPT,
        '--with_prior_preservation',
        '--prior_loss_weight',            str(PRIOR_LOSS_WEIGHT),
        '--num_class_images',             str(NUM_CLASS_IMAGES),
        '--resolution',                   '512',
        '--train_batch_size',             '1',
        '--gradient_accumulation_steps',  '4',
        '--gradient_checkpointing',
        '--max_train_steps',              str(MAX_TRAIN_STEPS),
        '--learning_rate',                str(LEARNING_RATE),
        '--text_encoder_lr',              str(TEXT_ENCODER_LR),
        '--lr_scheduler',                 'cosine_with_restarts',
        '--lr_warmup_steps',              '300',
        '--rank',                         str(LORA_RANK),
        '--train_text_encoder',
        '--mixed_precision',              'fp16',
        '--enable_xformers_memory_efficient_attention',
        '--seed',                         str(SEED),
        '--checkpointing_steps',          str(CHECKPOINT_STEPS),
        '--resume_from_checkpoint',       'latest',
        '--snr_gamma',                    str(SNR_GAMMA),
        '--random_flip',
        '--validation_prompt',            INSTANCE_PROMPT,
        '--num_validation_images',        '4',
        '--validation_steps',             '1000',
    ]

    print(f'Entrenando LoRA — rank={LORA_RANK}, steps={MAX_TRAIN_STEPS}')
    print(f'Checkpoint cada {CHECKPOINT_STEPS} steps en {LORA_DIR}')
    print(f'Resume automático si la sesión se interrumpe.\n')
    print('NOTA: si falla con "unrecognized argument --text_encoder_lr",')
    print('      quita esa línea del cmd[] y vuelve a correr.\n')

    t0     = time.time()
    result = subprocess.run(cmd)   # output fluye al notebook en tiempo real
    elapsed = time.time() - t0

    if result.returncode == 0:
        lora_done.write_text(json.dumps({
            'completed_at':      time.strftime('%Y-%m-%dT%H:%M:%S'),
            'steps':             MAX_TRAIN_STEPS,
            'rank':              LORA_RANK,
            'instance_prompt':   INSTANCE_PROMPT,
            'prior_preservation': True,
            'num_class_images':  NUM_CLASS_IMAGES,
            'snr_gamma':         SNR_GAMMA,
            'train_text_encoder': True,
            'elapsed_h':         round(elapsed / 3600, 2),
        }, indent=2))
        print(f'\n✅ LoRA completo en {elapsed/3600:.2f}h')
    else:
        print(f'\n❌ Exit code {result.returncode} — revisa los logs arriba')


In [ ]:
# ── Bloque 3 — Generar imágenes (resumible por número de archivos) ────────────
# Si la sesión se cierra durante la generación, vuelve a correr esta celda.
# Cuenta los archivos lora_*.jpg existentes y continúa desde ahí.

if gen_done.exists():
    n_done = len(list(SYNTH_DIR.glob('lora_*.jpg')))
    print(f'Generación completa ({n_done} imágenes) — saltando')
else:
    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
    from PIL import Image

    n_start = len(sorted(SYNTH_DIR.glob('lora_*.jpg')))
    print(f'Generando desde imagen {n_start} → {GENERATE_N}')
    print(f'Scheduler : DPM++ 2M Karras, {GENERATE_STEPS} steps, CFG={GENERATE_CFG}')
    print(f'Prompt    : {GENERATE_PROMPT}\n')

    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, safety_checker=None,
    ).to(DEVICE)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config,
        use_karras_sigmas=True,
        algorithm_type='dpmsolver++',
    )
    pipe.load_lora_weights(str(LORA_DIR))
    pipe.enable_xformers_memory_efficient_attention()
    pipe.set_progress_bar_config(disable=True)

    n_gen = n_start
    t0    = time.time()

    with torch.inference_mode():
        while n_gen < GENERATE_N:
            batch_n   = min(GENERATE_BATCH, GENERATE_N - n_gen)
            generator = torch.Generator(device=DEVICE).manual_seed(SEED + n_gen)
            images    = pipe(
                [GENERATE_PROMPT]  * batch_n,
                negative_prompt=[NEGATIVE_PROMPT] * batch_n,
                num_inference_steps=GENERATE_STEPS,
                guidance_scale=GENERATE_CFG,
                generator=generator,
            ).images
            for img in images:
                img.save(str(SYNTH_DIR / f'lora_{n_gen:05d}.jpg'), quality=95)
                n_gen += 1
            if n_gen % 500 == 0 or n_gen == GENERATE_N:
                mins   = (time.time() - t0) / 60
                speed  = (n_gen - n_start) / max(mins, 1e-3)
                remain = (GENERATE_N - n_gen) / max(speed, 1e-3)
                print(f'  {n_gen}/{GENERATE_N}  |  {speed:.0f} img/min  |  ~{remain:.0f} min restantes')

    gen_done.write_text(json.dumps({
        'n':                   n_gen,
        'prompt':              GENERATE_PROMPT,
        'negative_prompt':     NEGATIVE_PROMPT,
        'num_inference_steps': GENERATE_STEPS,
        'guidance_scale':      GENERATE_CFG,
        'scheduler':           'DPM++ 2M Karras',
        'lora_rank':           LORA_RANK,
        'lora_steps':          MAX_TRAIN_STEPS,
        'completed_at':        time.strftime('%Y-%m-%dT%H:%M:%S'),
    }, indent=2))

    del pipe
    torch.cuda.empty_cache()
    print(f'\n✅ {n_gen} imágenes guardadas en {SYNTH_DIR}')


In [ ]:
# ── Preview — muestra 8 muestras aleatorias ───────────────────────────────────
import matplotlib.pyplot as plt
import random
from PIL import Image

imgs = sorted(SYNTH_DIR.glob('lora_*.jpg'))
if not imgs:
    print('Sin imágenes — corre el bloque 3 primero')
else:
    sample = random.sample(imgs, min(8, len(imgs)))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, p in zip(axes.flat, sample):
        ax.imshow(Image.open(p).resize((256, 256)))
        ax.axis('off')
        ax.set_title(p.name, fontsize=7)
    plt.suptitle(
        f'LoRA melanoma  —  rank={LORA_RANK}, steps={MAX_TRAIN_STEPS}  '
        f'|  {len(imgs)} imágenes totales',
        fontsize=12
    )
    plt.tight_layout()
    out = SYNTH_DIR / 'preview_grid.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Preview guardado: {out}')
